In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import tensorflow as tf
from get_light_status import TrafficLightDetector
from sound_manager import TrafficSoundManager

E0000 00:00:1773677463.382734  138117 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773677463.386057  138117 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773677463.394358  138117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773677463.394367  138117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773677463.394368  138117 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773677463.394369  138117 computation_placer.cc:177] computation placer already registered. Please check linka

pygame 2.1.3 (SDL 2.0.22, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            # Enable memory growth so it doesn't seize the whole card
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(f"Memory growth error: {e}")

TensorFlow GPU Memory Growth Enabled


In [ ]:
detector = YOLO("yolo26m.pt") 
traffic_light_detector = TrafficLightDetector()
sound_manager = TrafficSoundManager()

In [ ]:
# cap = cv2.VideoCapture(0) # 0 is usually the built-in camera
cap = cv2.VideoCapture('video/5.mp4')


In [ ]:
frame_count = 0

OUTPUT_WIDTH = 1280
OUTPUT_HEIGHT = 720

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1

    if frame_count % 4 != 0:
        display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
        cv2.imshow('Traffic Light Monitor', display_frame) # Keep the window fluid
        if cv2.waitKey(1) & 0xFF == ord('q'): break
        continue

    

    # 3. Detection (Class 9 is traffic light)
    results = detector(frame, verbose=False, classes=[9])[0]
    
    lights = results.boxes
    if len(lights) > 0:
        # best_box = max(lights, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
        # for box in lights:

        
        # 1. Find the biggest box by calculating area: (x2 - x1) * (y2 - y1)
        best_box = max(lights, key=lambda b: (b.xyxy[0][2] - b.xyxy[0][0]) * (b.xyxy[0][3] - b.xyxy[0][1]))

        x1, y1, x2, y2 = best_box.xyxy[0].cpu().numpy().astype(int)

            # 2. Boundary-safe Cropping
        # Ensure coordinates are within frame limits
        crop = frame[max(0, y1):y2, max(0, x1):x2]
        
        if crop.size == 0:
            continue

            # 3. Direct Classification (No resizing unless get_light_state requires it)
        # Tip: Move cv2.resize INSIDE get_light_state only if absolutely necessary
        # color_label = get_light_state(crop)
        state = traffic_light_detector.get_light_state(crop)
        # 3. Handle Sound (The Class handles all the "if" logic internally)
        sound_manager.update(state)

        
        
        # 4. Immediate Drawing (Skips the list/enumerate overhead)
        # bgr_color = COLOR_MAP.get(color_label, (255, 255, 255))
        bgr_color = traffic_light_detector.COLOR_MAP.get(state, (255, 255, 255)) # Default to White if UNKNOWN

        cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
        # cv2.putText(frame, f"{color_label}", (x1, y1-10), 
        #         cv2.FONT_HERSHEY_SIMPLEX, 0.9, bgr_color, 2)
                
    display_frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT), interpolation=cv2.INTER_AREA)
    cv2.imshow('Traffic Light Monitor', display_frame)
    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
   

cap.release()
cv2.destroyAllWindows()
sound_manager.cleanup()
